# Climate GGMP (Eq. 11) — EM Weights End-to-End

This notebook runs the **full GGMP pipeline** on the climate station dataset:

1. **Station data → per-station ordered K-component Gaussians** (means/variances).
2. **Train K independent GPs** (one per component index) on the component-mean columns using **fvGP MCMC**.
3. **Optimize a shared weight vector** \(w_y = w_d = w\) using **EM** for the density-based Eq. 11 objective.
4. Evaluate on held-out test stations (metrics + plots).

## Sections
- 0. Config
- 1. Load + split data
- 2. Fit station mixtures (fixed weights)
- 3. Train GPs (MCMC)
- 4. EM weight optimization (density objective)
- 5. Test evaluation


## 0) Config

Tweak these to trade off speed vs quality. For a quick smoke test, keep `FAST_MODE=True`.

GPU:
- Set `USE_GPU=True` to request fvGP GPU compute paths.
- If no compatible backend is present, GGMP will fall back to CPU.


In [ ]:
from __future__ import annotations

import os
import time
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Single source of truth: all helpers live in ggmp.py
# Notebook dev tip: reload ggmp so edits apply without a full kernel restart.
import importlib
import ggmp as ggmp_mod
importlib.reload(ggmp_mod)

from ggmp import (
    GGMP,
    hyperparameters,
    _get_key,
    gaussian_pdf,
    empirical_pdf_from_samples,
    fit_station_gmms_fixed_weights_cached,
    build_gp_init_kwargs,
    _parse_device_ids,
    _normalize_pdf,
    train_component_gps_mcmc,
    prepare_station_terms_density,
    optimize_weights_em_density,
    bhattacharyya_distance,
    kl_divergence,
    wasserstein_1d,
)

# --------------------------
# Toggle-friendly parameters
# --------------------------
FAST_MODE = False
USE_ALL_STATIONS = True  # set True to use *all* valid stations (slow)

BASE_PATH = Path(".")  # expects data.npy + station_coord.npy here
OUT_DIR = Path("ggmp_em_runs")

SEED = 42
MIN_SAMPLES = 50
TRAIN_RATIO = 0.8
MAX_STATIONS = None if USE_ALL_STATIONS else (100 if FAST_MODE else 500)

K = 1
HIST_BINS = 120

# Station-level GMM fit
GMM_MAX_ITER = 100
GMM_TOL = 1e-4
CACHE_GMM = True
GMM_CACHE_DIR = Path("ggmp_cache")

# GP MCMC iterations per component
MCMC_UNTIL_CONVERGED = True
MCMC_CHUNK = 10 if FAST_MODE else 100
MCMC_MAX_TOTAL = 50 if FAST_MODE else 5000
MCMC_TOL_REL = 1e-2 if FAST_MODE else 1e-3
MCMC_PATIENCE = 2 if FAST_MODE else 5

# Parallelize training across components (threads)
GP_PARALLEL = None
GP_WORKERS = None  # None => use K workers
BLAS_THREADS_PER_GP = 1  # recommended when GP_PARALLEL=True

# Save fvGP MCMC traces (gp.mcmc_info)
SAVE_GP_MCMC = True
GP_MCMC_THIN = 1
SAVE_GP_MCMC_CHUNKS = True  # when MCMC_UNTIL_CONVERGED=True

# If MCMC_UNTIL_CONVERGED=False, use a fixed budget:
N_UPDATES_GP = 50 if FAST_MODE else 500

# EM weight fit
WEIGHT_FLOOR = 1e-6
EM_MAX_ITER = 80 if FAST_MODE else 200
EM_TOL_L1 = 1e-10
EM_LOG_EVERY = 5 if FAST_MODE else 10

# Plotting
N_PLOT_STATIONS = 8
PLOT_BINS = 30

# Threads (fvGP + BLAS)
N_THREADS = 1

# GPU options
USE_GPU = False
GPU_ENGINE = "torch"   # "torch" or "cupy"
GP_DEVICE_IDS = None    # None, "auto", "0,1", ...

# --------------------------
# Repro + logging
# --------------------------
np.random.seed(SEED)

os.environ["NTHREADS"] = str(int(N_THREADS))
os.environ["OPENBLAS_NUM_THREADS"] = str(int(N_THREADS))
os.environ["MKL_NUM_THREADS"] = str(int(N_THREADS))
os.environ["OMP_NUM_THREADS"] = str(int(N_THREADS))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

OUT_DIR.mkdir(parents=True, exist_ok=True)
run_dir = OUT_DIR / f"nb_run_{time.strftime('%Y%m%d_%H%M%S')}_K{K}"
run_dir.mkdir(parents=True, exist_ok=True)

logging.info("run_dir=%s", run_dir)
logging.info(
    "FAST_MODE=%s | K=%d | MAX_STATIONS=%s | MCMC_UNTIL_CONVERGED=%s | N_UPDATES_GP=%s | MCMC_CHUNK=%s | MCMC_MAX_TOTAL=%s",
    FAST_MODE,
    K,
    MAX_STATIONS,
    MCMC_UNTIL_CONVERGED,
    N_UPDATES_GP,
    MCMC_CHUNK,
    MCMC_MAX_TOTAL,
)

## 1) Load + split data

We split **by station id**: train stations define the GGMP training set, test stations are held out.


In [ ]:
# --- Load data ---
temp = np.load(str(BASE_PATH / "data.npy"))
coords = np.load(str(BASE_PATH / "station_coord.npy"))
lon = coords[:, 0]
lat = coords[:, 1]

valid_counts = np.sum(~np.isnan(temp), axis=0)
valid_stations = np.where(valid_counts >= int(MIN_SAMPLES))[0]
logging.info("Valid stations: %d (min_samples=%d)", len(valid_stations), MIN_SAMPLES)

if (MAX_STATIONS is not None) and (len(valid_stations) > int(MAX_STATIONS)):
    valid_stations = np.random.choice(valid_stations, size=int(MAX_STATIONS), replace=False)
    logging.info("Subsampled to MAX_STATIONS=%d", MAX_STATIONS)

if MAX_STATIONS is None:
    logging.info("Using all valid stations (no subsampling).")

n_train = int(len(valid_stations) * float(TRAIN_RATIO))
perm = np.random.permutation(len(valid_stations))
train_idx = np.asarray(valid_stations[perm[:n_train]], dtype=int)
test_idx = np.asarray(valid_stations[perm[n_train:]], dtype=int)

x_train = np.column_stack([lon[train_idx], lat[train_idx]])
x_test = np.column_stack([lon[test_idx], lat[test_idx]])

logging.info("Train stations: %d | Test stations: %d", len(train_idx), len(test_idx))

# Extract station series (drop NaNs)
def extract_series(ids):
    out = []
    for sid in ids:
        y = temp[:, int(sid)]
        y = y[~np.isnan(y)]
        out.append(np.asarray(y, dtype=float))
    return out

train_series = extract_series(train_idx)

logging.info("Example train series length: %d", len(train_series[0]) if train_series else -1)


In [ ]:
# Leakage sanity checks (pre-model)
assert np.intersect1d(train_idx, test_idx).size == 0
logging.info('Leakage checks passed: train/test station ids are disjoint.')


## 2) Fit station mixtures (fixed weights)

For each station we fit **ordered** \(K\)-component Gaussians (means/variances).

At this stage, we enforce a *shared* weight vector by holding weights fixed (initially uniform). We will fit the optimal shared weights \(w\) later via EM.


In [ ]:
w_init = np.ones(int(K), dtype=float) / float(K)

means_mat, vars_mat, gmm_cache_path = fit_station_gmms_fixed_weights_cached(
    train_series,
    train_idx,
    data_path=BASE_PATH / 'data.npy',
    K=int(K),
    gmm_max_iter=int(GMM_MAX_ITER),
    gmm_tol=float(GMM_TOL),
    cache=bool(CACHE_GMM),
    cache_dir=GMM_CACHE_DIR,
    log_every=100,
)

if gmm_cache_path is not None:
    logging.info('GMM cache path: %s', gmm_cache_path)

means_mat.shape, vars_mat.shape


### 2a) Marginalization verification plot (using current weights)

This checks that the per-station fitted components marginalize back to something close to each station’s empirical histogram.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

sample_indices = np.random.choice(len(train_idx), size=min(int(N_PLOT_STATIONS), len(train_idx)), replace=False)

for plot_idx, i in enumerate(sample_indices):
    ax = axes[plot_idx]

    y = train_series[i]
    means = means_mat[i]
    vars_ = vars_mat[i]

    ax.hist(y, bins=int(PLOT_BINS), density=True, alpha=0.5, label='Empirical')

    x_range = np.linspace(float(y.min() - 2), float(y.max() + 2), 200)
    gmm_pdf = np.zeros_like(x_range)
    for k in range(int(K)):
        component_pdf = float(w_init[k]) * gaussian_pdf(x_range, float(means[k]), float(vars_[k]))
        gmm_pdf += component_pdf
        ax.plot(x_range, component_pdf, '--', alpha=0.5, label=f'Comp {k}')

    ax.plot(x_range, gmm_pdf, 'r-', linewidth=2, label='GMM')
    ax.set_title(f'Station {int(train_idx[i])}')
    ax.legend(fontsize=6)

plt.tight_layout()
plt.show()


## 3) Train GPs (fvGP MCMC)

We train one GP per component index \(k\), using:
- targets: the column `means_mat[:, k]`
- noise variances: `vars_mat[:, k]`

Each GP is trained independently with `method='mcmc'`.


In [ ]:
# Observed station PDFs (train) as empirical histograms
y_data_train = [empirical_pdf_from_samples(y, bins=int(HIST_BINS)) for y in train_series]

# GGMP hyperparameters setup
weights_bounds = np.array([[float(WEIGHT_FLOOR), 1.0]] * int(K), dtype=float)

hps_list = []
hps_bounds_list = []
for k in range(int(K)):
    mean_k = float(np.mean(means_mat[:, k]))
    hps_list.append(np.array([1.0, 1.0, 1.0, mean_k], dtype=float))  # [signal, L_lon, L_lat, mean]
    hps_bounds_list.append(
        np.array(
            [
                [0.1, 10.0],
                [0.01, 100.0],
                [0.01, 100.0],
                [mean_k - 40.0, mean_k + 40.0],
            ],
            dtype=float,
        )
    )

hps_obj = hyperparameters(w_init.copy(), weights_bounds, hps_list, hps_bounds_list)

# fvGP GPU toggles (passed via GGMP.initGPs)
gp_init_kwargs, _ = build_gp_init_kwargs(use_gpu=USE_GPU, gpu_engine=GPU_ENGINE)

model = GGMP(
    x_train,
    y_data_train,
    hps_obj=hps_obj,
    likelihood_terms=int(K),
    gp_init_kwargs=gp_init_kwargs,
    gp_device_ids=_parse_device_ids(GP_DEVICE_IDS),
)

init_mean = [means_mat[:, k] for k in range(int(K))]
init_std = [np.sqrt(vars_mat[:, k]) for k in range(int(K))]
model.initLikelihoods(init_mean=init_mean, init_std=init_std, weights=w_init)
model.initGPs()

logging.info("Initialized GGMP: N=%d, K=%d", model.len_data, model.likelihood_terms)


# Leakage sanity checks (post-model)
assert model.x_data.shape == x_train.shape and np.allclose(model.x_data, x_train)
assert model.len_data == len(train_idx)
assert len(model.y_data) == len(train_idx)
logging.info('Leakage checks passed: GGMP model is built only from train data.')


In [ ]:
trained_hps = train_component_gps_mcmc(
    model,
    hps_obj,
    n_updates_gp=int(N_UPDATES_GP),
    mcmc_until_converged=bool(MCMC_UNTIL_CONVERGED),
    mcmc_chunk=int(MCMC_CHUNK),
    mcmc_max_total=int(MCMC_MAX_TOTAL),
    mcmc_tol_rel=float(MCMC_TOL_REL),
    mcmc_patience=int(MCMC_PATIENCE),
    gp_parallel=bool(GP_PARALLEL),
    gp_workers=(int(GP_WORKERS) if GP_WORKERS is not None else None),
    blas_threads_per_gp=(int(BLAS_THREADS_PER_GP) if BLAS_THREADS_PER_GP is not None else None),
    run_dir=run_dir,
    save_gp_mcmc=bool(SAVE_GP_MCMC),
    gp_mcmc_thin=int(GP_MCMC_THIN),
    save_gp_mcmc_chunks=bool(SAVE_GP_MCMC_CHUNKS),
)

trained_hps

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Optional override if you want a specific run directory.
RUN_DIR_OVERRIDE = None  # e.g. Path("ggmp_em_runs/nb_run_YYYYMMDD_HHMMSS_K1")

base_runs = OUT_DIR if 'OUT_DIR' in globals() else Path('ggmp_em_runs')
if RUN_DIR_OVERRIDE is not None:
    RUN_DIR = Path(RUN_DIR_OVERRIDE)
else:
    candidates = sorted(base_runs.glob('nb_run_*_K*'))
    if not candidates:
        raise FileNotFoundError(f'No run directories found under {base_runs}')
    RUN_DIR = candidates[-1]

BURN_FRAC = 0.3

def _load_npz(p: Path):
    z = np.load(p, allow_pickle=False)
    x = np.asarray(z["x"], dtype=float)
    meta = json.loads(str(z["meta_json"])) if "meta_json" in z.files else {}
    fx = np.asarray(z["f"], dtype=float) if "f" in z.files else None
    return x, fx, meta

def _plot_trace(k: int, X: np.ndarray, FX=None, title_extra=""):
    n, d = X.shape
    burn = int(BURN_FRAC * n)

    nrows = d + (1 if FX is not None else 0)
    fig, axes = plt.subplots(nrows, 1, figsize=(11, 2.2 * nrows), sharex=True)
    axes = np.atleast_1d(axes)

    for j in range(d):
        axes[j].plot(X[:, j], lw=0.8)
        axes[j].axvline(burn, color="r", lw=1, alpha=0.6)
        axes[j].set_ylabel(f"hps[{j}]")
        axes[j].grid(True, alpha=0.25)

    if FX is not None:
        axes[-1].plot(FX, lw=0.8, color="k")
        axes[-1].axvline(burn, color="r", lw=1, alpha=0.6)
        axes[-1].set_ylabel("f(x)")
        axes[-1].grid(True, alpha=0.25)

    axes[-1].set_xlabel("sample")
    fig.suptitle(f"GP[{k}] MCMC trace {title_extra} | samples={n} dim={d}", y=0.98)
    plt.tight_layout()
    plt.show()

print("RUN_DIR:", RUN_DIR.resolve())

trace_files = sorted(RUN_DIR.glob("gp*_mcmc_trace*.npz")) + sorted((RUN_DIR / "gp_mcmc").glob("gp*_mcmc_trace*.npz"))
trace_files = sorted({p.resolve() for p in trace_files})

print("Found trace files:", len(trace_files))
for p in trace_files[:30]:
    print(" -", p.name)

by_k = {}
for p in trace_files:
    try:
        _, _, meta = _load_npz(p)
        k = int(meta.get("k"))
    except Exception:
        k = int(p.name[p.name.find("gp")+2:p.name.find("gp")+4])
    by_k.setdefault(k, []).append(p)

for k, files in sorted(by_k.items()):
    chunk_files = sorted([p for p in files if "chunk" in p.name])
    if chunk_files:
        Xs, FXs = [], []
        for p in chunk_files:
            X, FX, _ = _load_npz(p)
            Xs.append(X)
            if FX is not None:
                FXs.append(FX)
        Xfull = np.vstack(Xs)
        FXfull = np.concatenate(FXs) if (FXs and len(FXs) == len(Xs)) else None
        _plot_trace(k, Xfull, FXfull, title_extra="(concatenated chunks)")
    else:
        p = sorted(files)[-1]
        X, FX, _ = _load_npz(p)
        _plot_trace(k, X, FX, title_extra=f"({p.name})")


## 4) EM weight optimization (density objective)

We optimize shared weights \(w\) for the density-based Eq. 11 objective:

$$
\sum_{i=1}^N \int p_{	ext{obs},i}(y)\; \log\Big( \sum_{k=1}^K w_k\;\mathcal{N}(y\mid \mu_{ik},\,\sigma^2_{ik}) \Big)\;dy
$$

We approximate each integral by a discrete sum on a histogram grid, and run EM on \(w\).


In [ ]:
terms, ll_comp = prepare_station_terms_density(model, trained_hps)
logging.info("Expected component-only log-likelihoods: %s", np.array2string(ll_comp, precision=3))

w_em, w_hist, obj_hist = optimize_weights_em_density(
    terms,
    K=int(K),
    weight_floor=float(WEIGHT_FLOOR),
    max_iter=int(EM_MAX_ITER),
    tol_l1=float(EM_TOL_L1),
    log_every=int(EM_LOG_EVERY),
    w0=w_init,
)

logging.info("Final EM weights: %s", np.array2string(w_em, precision=6))
logging.info("Final objective: %.6f", float(obj_hist[-1]) if obj_hist.size else float('nan'))

# Plot traces
fig = plt.figure(figsize=(10, 4))
ax1 = fig.add_subplot(1, 2, 1)
ax1.plot(obj_hist, '-k')
ax1.set_title('EM objective')
ax1.set_xlabel('iteration')
ax1.set_ylabel('objective')
ax2 = fig.add_subplot(1, 2, 2)
for k in range(int(K)):
    ax2.plot(w_hist[:, k], label=f'w[{k}]')
ax2.set_title('EM weights')
ax2.set_xlabel('iteration')
ax2.set_ylabel('weight')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()


### 4a) Marginalization verification plot (using optimized weights)

Same plot as before, but using $w_{EM}$.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

sample_indices = np.random.choice(len(train_idx), size=min(int(N_PLOT_STATIONS), len(train_idx)), replace=False)

for plot_idx, i in enumerate(sample_indices):
    ax = axes[plot_idx]

    y = train_series[i]
    means = means_mat[i]
    vars_ = vars_mat[i]

    ax.hist(y, bins=int(PLOT_BINS), density=True, alpha=0.5, label='Empirical')

    x_range = np.linspace(float(y.min() - 2), float(y.max() + 2), 200)
    gmm_pdf = np.zeros_like(x_range)
    for k in range(int(K)):
        component_pdf = float(w_em[k]) * gaussian_pdf(x_range, float(means[k]), float(vars_[k]))
        gmm_pdf += component_pdf
        ax.plot(x_range, component_pdf, '--', alpha=0.5, label=f'Comp {k}')

    ax.plot(x_range, gmm_pdf, 'r-', linewidth=2, label='GMM')
    ax.set_title(f'Station {int(train_idx[i])}')
    ax.legend(fontsize=6)

plt.tight_layout()
plt.show()


## 5) Test evaluation

We predict a distribution for each **held-out station** and compare to its empirical histogram.

Metrics:
- Bhattacharyya distance
- symmetric KL $$\frac{1}{2}(KL(p\|q)+KL(q\|p))\$$
- 1D Wasserstein-1
- L1 distance


In [ ]:
# Update model weights so posterior mixing uses w_em
model.hps_obj.set(w_em, trained_hps)
for k in range(int(K)):
    try:
        model.likelihoods[k].set_weight(float(w_em[k]))
    except Exception:
        pass

# IMPORTANT (leakage guard): only materialize test station values here (after training).
test_series = extract_series(test_idx)

# Empirical PDFs for test stations
y_data_test = [empirical_pdf_from_samples(y, bins=int(HIST_BINS)) for y in test_series]

# GP predictive mean/variance for component means at test stations
mu_test = np.empty((len(test_idx), int(K)), dtype=float)
var_gp_test = np.empty((len(test_idx), int(K)), dtype=float)

for k in range(int(K)):
    gp = model.gps[k]
    model._safe_set_hyperparameters(gp, trained_hps[k])
    pm = gp.posterior_mean(x_test)
    pc = gp.posterior_covariance(x_test, variance_only=True)
    mu_test[:, k] = np.asarray(_get_key(pm, ('m(x)', 'f(x)')), dtype=float).reshape(-1)
    var_gp_test[:, k] = np.maximum(np.asarray(_get_key(pc, ('v(x)', 'v', 'variance', 'cov')), dtype=float).reshape(-1), 0.0)

# Use global within-component variance (mean over train stations)
var_comp_global = np.array([float(np.mean(model.likelihoods[k].variance)) for k in range(int(K))], dtype=float)
var_comp_global = np.maximum(var_comp_global, 1e-9)

metrics = []
for j, (domain, density) in enumerate(y_data_test):
    domain, p_obs, dx = _normalize_pdf(domain, density)

    mix = np.zeros_like(domain, dtype=float)
    for k in range(int(K)):
        sigma = float(np.sqrt(max(var_gp_test[j, k] + var_comp_global[k], 1e-12)))
        mix += float(w_em[k]) * norm.pdf(domain, loc=float(mu_test[j, k]), scale=sigma)

    mix = np.maximum(mix, 1e-300)
    mix = mix / float(np.sum(mix * dx) + 1e-300)

    bdist = bhattacharyya_distance(domain, p_obs, mix)
    kl_pq = kl_divergence(domain, p_obs, mix)
    kl_qp = kl_divergence(domain, mix, p_obs)
    w1 = wasserstein_1d(domain, p_obs, mix)
    l1 = float(np.sum(np.abs(p_obs - mix) * dx))

    metrics.append({
        'station_id': int(test_idx[j]),
        'bhattacharyya': float(bdist),
        'kl_sym': float(0.5 * (kl_pq + kl_qp)),
        'wasserstein1': float(w1),
        'l1': float(l1),
    })

# Summaries
for key in ('bhattacharyya', 'kl_sym', 'wasserstein1', 'l1'):
    arr = np.asarray([m[key] for m in metrics], dtype=float)
    logging.info("TEST %s: mean=%.4f | median=%.4f | std=%.4f", key, float(np.mean(arr)), float(np.median(arr)), float(np.std(arr)))

metrics[:3]


In [ ]:
# Metric histograms
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
keys = ['bhattacharyya', 'kl_sym', 'wasserstein1', 'l1']

for ax, key in zip(axes, keys):
    vals = np.asarray([m[key] for m in metrics], dtype=float)
    ax.hist(vals, bins=30, alpha=0.8)
    ax.set_title(f"Test {key}")

plt.tight_layout()
plt.show()


In [ ]:
# Plot a few test stations: empirical hist vs predicted mixture
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

if len(test_idx) > 0:
    sample_test = np.random.choice(len(test_idx), size=min(int(N_PLOT_STATIONS), len(test_idx)), replace=False)
    for plot_idx, j in enumerate(sample_test):
        ax = axes[plot_idx]
        y = test_series[j]
        ax.hist(y, bins=int(PLOT_BINS), density=True, alpha=0.5, label='Empirical')

        x_range = np.linspace(float(np.min(y) - 2), float(np.max(y) + 2), 200)
        mix_pdf = np.zeros_like(x_range)
        for k in range(int(K)):
            sigma = float(np.sqrt(max(var_gp_test[j, k] + var_comp_global[k], 1e-12)))
            comp_pdf = float(w_em[k]) * norm.pdf(x_range, loc=float(mu_test[j, k]), scale=sigma)
            mix_pdf += comp_pdf
            ax.plot(x_range, comp_pdf, '--', alpha=0.5, label=f'Comp {k}')

        ax.plot(x_range, mix_pdf, 'r-', linewidth=2, label='Pred Mix')
        ax.set_title(f"Test Station {int(test_idx[j])}")
        ax.legend(fontsize=6)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import json

plot_js = [55, 22, 34, 15]
plot_js = [j for j in plot_js if j < len(test_idx)]

payload = {
    "K": int(K),
    "test_js": plot_js,
    "test_idx_vals": [int(test_idx[j]) for j in plot_js],
    "components": {}
}

for j in plot_js:
    wk = np.asarray(w_em, dtype=float)  # (K,)
    muk = np.asarray(mu_test[j, :], dtype=float)  # (K,)
    var_total = np.asarray(var_gp_test[j, :] + var_comp_global[:], dtype=float)  # (K,)
    var_total = np.maximum(var_total, 1e-12)

    payload["components"][str(j)] = {
        "w": wk.tolist(),
        "mu": muk.tolist(),
        "var": var_total.tolist(),
    }

print(json.dumps(payload, indent=2))


## 6) Save artifacts (optional)

This writes the same artifacts the script produces: EM traces, test metrics, and a few PNGs.


In [ ]:
import csv

# Save EM traces
np.save(run_dir / 'weights_em_history.npy', w_hist)
np.save(run_dir / 'objective_em_history.npy', obj_hist)

# Save test metrics
metrics_path = run_dir / 'test_metrics.csv'
with open(metrics_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metrics[0].keys()))
    writer.writeheader()
    writer.writerows(metrics)

logging.info('Saved %s', metrics_path)
